# Real-Time Streaming Responses with LLMs

Welcome to the **Streaming Responses** hands-on lesson! In standard generation, you send a prompt and wait silently until the full response is generated. With **streaming**, response tokens/chunks are transmitted as soon as they are computed by the model.

### What You Will Learn:
1. **Why Streaming Matters:** Understanding first-token latency vs total round-trip latency.
2. **Direct SDK Streaming:** How to stream responses chunk-by-chunk using Google Gemini (`generate_content_stream`).
3. **LangChain & OpenRouter Streaming:** Streaming chunks through `ChatOpenAI`.
4. **Offline Mock Simulation:** Test streaming concepts without needing an active API key or internet connection.
5. **Error Handling & Interruptions:** Gracefully handling network drops or client cancellations mid-stream.

## 1. Setup & Key Validation
Load your keys safely from your `.env` file.

In [ ]:
import os
import time
import sys
from dotenv import load_dotenv

load_dotenv()

gemini_key = os.getenv("GEMINI_API_KEY")
openrouter_key = os.getenv("OPENROUTER_API_KEY")

print(f"Gemini Key configured: {bool(gemini_key and gemini_key != 'your_gemini_api_key_here')}")
print(f"OpenRouter Key configured: {bool(openrouter_key and openrouter_key != 'your_openrouter_api_key_here')}")

## 2. Offline Simulation: How Streaming Works Mechanically
Before making cloud API calls, let's observe streaming mechanics locally. Notice how words arrive incrementally, creating an interactive user experience.

In [ ]:
def simulate_token_stream(text: str, delay_seconds: float = 0.05):
    """Yields words one-by-one to simulate tokens arriving over the wire."""
    tokens = text.split(" ")
    for token in tokens:
        time.sleep(delay_seconds)
        yield token + " "

sample_story = (
    "Streaming transforms the user experience. Instead of waiting several seconds "
    "for a complete reply, the user sees initial feedback in milliseconds. "
    "This drastically reduces perceived latency."
)

print("--- Starting Local Simulated Stream ---")
start_t = time.perf_counter()
first_token_t = None

for chunk in simulate_token_stream(sample_story, delay_seconds=0.08):
    if first_token_t is None:
        first_token_t = time.perf_counter()
    print(chunk, end="", flush=True)

total_t = time.perf_counter() - start_t
print(f"\n\n[Time to First Token: {(first_token_t - start_t)*1000:.1f}ms | Total Elapsed: {total_t:.2f}s]")

## 3. Streaming with Google Gemini (`google-genai`)
Using Google's unified `google-genai` SDK with `client.models.generate_content_stream`.

In [ ]:
if gemini_key and gemini_key != "your_gemini_api_key_here":
    from google import genai
    
    client = genai.Client(api_key=gemini_key)
    model_id = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
    prompt = "Explain how neural networks learn in three short bullet points."
    
    print(f"Streaming from Gemini ({model_id})...\n")
    start_time = time.perf_counter()
    first_token_time = None
    chunk_count = 0
    
    response_stream = client.models.generate_content_stream(
        model=model_id,
        contents=prompt
    )
    
    for chunk in response_stream:
        if first_token_time is None:
            first_token_time = time.perf_counter()
        if chunk.text:
            print(chunk.text, end="", flush=True)
            chunk_count += 1
            
    total_time = time.perf_counter() - start_time
    ttft = (first_token_time - start_time) * 1000 if first_token_time else 0
    print(f"\n\n--- Stats: {chunk_count} chunks received | TTFT: {ttft:.1f}ms | Total: {total_time:.2f}s ---")
else:
    print("GEMINI_API_KEY not configured. Set it in .env to run this cell.")

## 4. Streaming with LangChain & OpenRouter
LangChain provides a `.stream()` method on all ChatModel classes.

In [ ]:
if openrouter_key and openrouter_key != "your_openrouter_api_key_here":
    from langchain_openai import ChatOpenAI
    
    # Verified free model on OpenRouter as of 2026-10-30
    model_name = os.getenv("OPENROUTER_MODEL", "google/gemma-4-31b-it:free")
    
    llm = ChatOpenAI(
        model=model_name,
        base_url="https://openrouter.ai/api/v1",
        api_key=openrouter_key
    )
    
    print(f"Streaming via LangChain from OpenRouter ({model_name})...\n")
    for chunk in llm.stream("What is the difference between synchronous and asynchronous code?"):
        print(chunk.content, end="", flush=True)
    print("\n\n--- Stream complete ---")
else:
    print("OPENROUTER_API_KEY not configured. Set it in .env to run this cell.")

## 5. Exercises & Next Steps
1. **Buffer chunks:** Modify the streaming loop to buffer text until a full sentence ending (`.`, `!`, `?`) is reached before printing.
2. **Cancellation test:** Wrap the stream in a loop that terminates immediately if a specific stopword appears in the output.
3. **Next Lesson:** Head over to `06_streaming_and_batch/` and `projects/04_streaming_chat/` for complete terminal applications with latency benchmarks!